In [1]:
import os
import shutil
import requests
import tempfile
from tqdm import tqdm
from typing import IO
from pathlib import Path
from mindspore import context
context.set_context(mode=context.GRAPH_MODE, device_target="Ascend", device_id=0)

# 指定保存路径为 `home_path/.mindspore_examples`
cache_dir = Path.home() / '.mindspore_examples'

def http_get(url: str, temp_file: IO):
    """使用requests库下载数据，并使用tqdm库进行流程可视化"""
    req = requests.get(url, stream=True)
    content_length = req.headers.get('Content-Length')
    total = int(content_length) if content_length is not None else None
    progress = tqdm(unit='B', total=total)
    for chunk in req.iter_content(chunk_size=1024):
        if chunk:
            progress.update(len(chunk))
            temp_file.write(chunk)
    progress.close()

def download(file_name: str, url: str):
    """下载数据并存为指定名称"""
    if not os.path.exists(cache_dir):
        os.makedirs(cache_dir)
    cache_path = os.path.join(cache_dir, file_name)
    cache_exist = os.path.exists(cache_path)
    if not cache_exist:
        with tempfile.NamedTemporaryFile() as temp_file:
            http_get(url, temp_file)
            temp_file.flush()
            temp_file.seek(0)
            with open(cache_path, 'wb') as cache_file:
                shutil.copyfileobj(temp_file, cache_file)
    return cache_path

/usr/local/Ascend/ascend-toolkit/7.0.RC1/python/site-packages/tbe/tvm/contrib/ccec.py:766: DeprecationWarning: invalid escape sequence \L
  if not dirpath.find("AppData\Local\Temp"):
/usr/local/Ascend/ascend-toolkit/latest/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:222: DeprecationWarning: invalid escape sequence \B
  """
/usr/local/Ascend/ascend-toolkit/latest/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:140: DeprecationWarning: invalid escape sequence \c
  """
/home/ma-user/anaconda3/envs/MindSpore/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/ma-user/anaconda3/envs/MindSpore/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, 

In [2]:
#设置no_proxy环境变量
%env no_proxy='a.test.com,127.0.0.1,2.2.2.2'
imdb_path = download('aclImdb_v1.tar.gz', 'https://mindspore-website.obs.myhuaweicloud.com/notebook/datasets/aclImdb_v1.tar.gz')
imdb_path

env: no_proxy='a.test.com,127.0.0.1,2.2.2.2'


100%|██████████| 84125825/84125825 [00:02<00:00, 35109371.04B/s]


'/home/ma-user/.mindspore_examples/aclImdb_v1.tar.gz'

In [3]:
import re
import six
import string
import tarfile

class IMDBData():
    """IMDB数据集加载器

    加载IMDB数据集并处理为一个Python迭代对象。

    """
    label_map = {
        "pos": 1,
        "neg": 0
    }
    def __init__(self, path, mode="train"):
        self.mode = mode
        self.path = path
        self.docs, self.labels = [], []

        self._load("pos")
        self._load("neg")

    def _load(self, label):
        pattern = re.compile(r"aclImdb/{}/{}/.*\.txt$".format(self.mode, label))
        # 将数据加载至内存
        with tarfile.open(self.path) as tarf:
            tf = tarf.next()
            while tf is not None:
                if bool(pattern.match(tf.name)):
                    # 对文本进行分词、去除标点和特殊字符、小写处理
                    self.docs.append(str(tarf.extractfile(tf).read().rstrip(six.b("\n\r"))
                                         .translate(None, six.b(string.punctuation)).lower()).split())
                    self.labels.append([self.label_map[label]])
                tf = tarf.next()

    def __getitem__(self, idx):
        return self.docs[idx], self.labels[idx]

    def __len__(self):
        return len(self.docs)

In [4]:
imdb_train = IMDBData(imdb_path, 'train')
len(imdb_train)

25000

In [9]:
import mindspore.dataset as dataset  # 确保这里导入的名字和下面函数里用的一致

def load_imdb(imdb_path):
    # 手册里这里写的是 dataset.GeneratorDataset，所以上面必须 import ... as dataset 
    imdb_train = dataset.GeneratorDataset(IMDBData(imdb_path, "train"), column_names=["text", "label"], shuffle=True)
    imdb_test = dataset.GeneratorDataset(IMDBData(imdb_path, "test"), column_names=["text", "label"], shuffle= False)
    return imdb_train, imdb_test

In [11]:
imdb_train, imdb_test = load_imdb(imdb_path)
imdb_train

In [12]:
import zipfile
import numpy as np

def load_glove(glove_path):
    glove_100d_path = os.path.join(cache_dir, 'glove.6B.100d.txt')
    if not os.path.exists(glove_100d_path):
        glove_zip = zipfile.ZipFile(glove_path)
        glove_zip.extractall(cache_dir)

    embeddings = []
    tokens = []
    with open(glove_100d_path, encoding='utf-8') as gf:
        for glove in gf:
            word, embedding = glove.split(maxsplit=1)
            tokens.append(word)
            embeddings.append(np.fromstring(embedding, dtype=np.float32, sep=' '))
    # 添加 <unk>, <pad> 两个特殊占位符对应的embedding
    embeddings.append(np.random.rand(100))
    embeddings.append(np.zeros((100,), np.float32))

    vocab = dataset.text.Vocab.from_list(tokens, special_tokens=["<unk>", "<pad>"], special_first=False)
    embeddings = np.array(embeddings).astype(np.float32)
    return vocab, embeddings

In [13]:
glove_path = download('glove.6B.zip', 'https://mindspore-website.obs.myhuaweicloud.com/notebook/datasets/glove.6B.zip')
vocab, embeddings = load_glove(glove_path)
len(vocab.vocab())

100%|██████████| 862182613/862182613 [00:21<00:00, 40359416.93B/s]


400002

In [14]:
idx = vocab.tokens_to_ids('the')
embedding = embeddings[idx]
idx, embedding

(0,
 array([-0.038194, -0.24487 ,  0.72812 , -0.39961 ,  0.083172,  0.043953,
        -0.39141 ,  0.3344  , -0.57545 ,  0.087459,  0.28787 , -0.06731 ,
         0.30906 , -0.26384 , -0.13231 , -0.20757 ,  0.33395 , -0.33848 ,
        -0.31743 , -0.48336 ,  0.1464  , -0.37304 ,  0.34577 ,  0.052041,
         0.44946 , -0.46971 ,  0.02628 , -0.54155 , -0.15518 , -0.14107 ,
        -0.039722,  0.28277 ,  0.14393 ,  0.23464 , -0.31021 ,  0.086173,
         0.20397 ,  0.52624 ,  0.17164 , -0.082378, -0.71787 , -0.41531 ,
         0.20335 , -0.12763 ,  0.41367 ,  0.55187 ,  0.57908 , -0.33477 ,
        -0.36559 , -0.54857 , -0.062892,  0.26584 ,  0.30205 ,  0.99775 ,
        -0.80481 , -3.0243  ,  0.01254 , -0.36942 ,  2.2167  ,  0.72201 ,
        -0.24978 ,  0.92136 ,  0.034514,  0.46745 ,  1.1079  , -0.19358 ,
        -0.074575,  0.23353 , -0.052062, -0.22044 ,  0.057162, -0.15806 ,
        -0.30798 , -0.41625 ,  0.37972 ,  0.15006 , -0.53212 , -0.2055  ,
        -1.2526  ,  0.071624,  0.7

In [17]:
import mindspore as ms

# 使用更现代的路径（去掉 .c_transforms）
lookup_op = dataset.text.Lookup(vocab, unknown_token='<unk>')
pad_op = dataset.transforms.PadEnd([500], pad_value=vocab.tokens_to_ids('<pad>'))
type_cast_op = dataset.transforms.TypeCast(ms.float32)


In [20]:
# 这一步确保 imdb_train 回到初始状态 [cite: 215]
imdb_train, imdb_test = load_imdb(imdb_path)

In [21]:
# 必须运行这两个 map，数据集才会具备后续操作的能力 [cite: 283-284]
imdb_train = imdb_train.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_train = imdb_train.map(operations=[type_cast_op], input_columns=['label'])

imdb_test = imdb_test.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_test = imdb_test.map(operations=[type_cast_op], input_columns=['label'])

In [22]:
imdb_train, imdb_valid = imdb_train.split([0.7, 0.3])

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-20:48:00.393.887 [mindspore/dataset/engine/datasets.py:1203] Dataset is shuffled before split.


In [23]:
imdb_train = imdb_train.batch(64, drop_remainder=True)
imdb_valid = imdb_valid.batch(64, drop_remainder=True)

In [25]:
import mindspore as ms
import mindspore.dataset as dataset # 加上这一行，确保 dataset 被定义 

# 接下来再运行预处理操作 [cite: 279-281]
lookup_op = dataset.text.Lookup(vocab, unknown_token='<unk>')
pad_op = dataset.transforms.PadEnd([500], pad_value=vocab.tokens_to_ids('<pad>'))
type_cast_op = dataset.transforms.TypeCast(ms.float32)

print("预处理操作定义成功！")

预处理操作定义成功！


In [26]:
# 应用 map 操作 [cite: 283-286]
imdb_train = imdb_train.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_train = imdb_train.map(operations=[type_cast_op], input_columns=['label'])
imdb_test = imdb_test.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_test = imdb_test.map(operations=[type_cast_op], input_columns=['label'])

# 拆分验证集 [cite: 287-288]
imdb_train, imdb_valid = imdb_train.split([0.7, 0.3])

# 打包为 Batch [cite: 290-291]
imdb_train = imdb_train.batch(64, drop_remainder=True)
imdb_valid = imdb_valid.batch(64, drop_remainder=True)

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-20:51:27.183.590 [mindspore/dataset/engine/datasets.py:1203] Dataset is shuffled before split.


In [35]:
import mindspore.nn as nn
import mindspore.ops as ops
import mindspore.numpy as mnp
from mindspore import Tensor

class RNN(nn.Cell):
    def __init__(self, embeddings, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx):
        super().__init__() # 必须有这一行！ 
        vocab_size, embedding_dim = embeddings.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, 
                                     embedding_table=Tensor(embeddings), 
                                     padding_idx=pad_idx)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                          bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Dense(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(1 - dropout)
        self.sigmoid = ops.Sigmoid()

    def construct(self, inputs):
        embedded = self.dropout(self.embedding(inputs))
        _, (hidden, _) = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(mnp.concatenate((hidden[-2, :, :], hidden[-1, :, :]), axis=1))
        else:
            hidden = self.dropout(hidden[-1, :, :])
        return self.sigmoid(self.fc(hidden))

In [38]:
# 参数设置
hidden_size = 256
output_size = 1
num_layers = 2
bidirectional = True
lr = 0.001
pad_idx = vocab.tokens_to_ids('<pad>')

# 实例化模型
model = RNN(embeddings, hidden_size, output_size, num_layers, bidirectional, 0.5, pad_idx)

# 定义损失函数和优化器 [cite: 350-351]
loss_fn = nn.BCEWithLogitsLoss(reduction='mean')
optimizer = nn.Adam(model.trainable_params(), learning_rate=lr)

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:02:25.373.753 [mindspore/nn/layer/basic.py:173] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


In [39]:
def forward_fn(data, label):
    logits = model(data)
    loss = loss_fn(logits, label)
    return loss

grad_fn = ms.value_and_grad(forward_fn, None, optimizer.parameters)

def train_step(data, label):
    loss, grads = grad_fn(data, label)
    optimizer(grads)
    return loss

def train_one_epoch(model, train_dataset, epoch=0):
    model.set_train()
    total = train_dataset.get_dataset_size()
    loss_total = 0
    step_total = 0
    with tqdm(total=total) as t:
        t.set_description('Epoch %i' % epoch)
        for i in train_dataset.create_tuple_iterator():
            loss = train_step(*i)
            loss_total += loss.asnumpy()
            step_total += 1
            t.set_postfix(loss=loss_total/step_total)
            t.update(1)

In [40]:
def binary_accuracy(preds, y):
    """
    计算每个batch的准确率
    """

    # 对预测值进行四舍五入
    rounded_preds = np.around(preds)
    correct = (rounded_preds == y).astype(np.float32)
    acc = correct.sum() / len(correct)
    return acc

In [41]:
def evaluate(model, test_dataset, criterion, epoch=0):
    total = test_dataset.get_dataset_size()
    epoch_loss = 0
    epoch_acc = 0
    step_total = 0
    model.set_train(False)

    with tqdm(total=total) as t:
        t.set_description('Epoch %i' % epoch)
        for i in test_dataset.create_tuple_iterator():
            predictions = model(i[0])
            loss = criterion(predictions, i[1])
            epoch_loss += loss.asnumpy()

            acc = binary_accuracy(predictions.asnumpy(), i[1].asnumpy())
            epoch_acc += acc

            step_total += 1
            t.set_postfix(loss=epoch_loss/step_total, acc=epoch_acc/step_total)
            t.update(1)

    return epoch_loss / total

In [43]:
# 1. 重新加载原始数据集（把数据变回字符串） [cite: 215]
imdb_train, imdb_test = load_imdb(imdb_path)

# 2. 重新应用预处理（只运行这一次！） [cite: 283-286]
imdb_train = imdb_train.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_train = imdb_train.map(operations=[type_cast_op], input_columns=['label'])
imdb_test = imdb_test.map(operations=[lookup_op, pad_op], input_columns=['text'])
imdb_test = imdb_test.map(operations=[type_cast_op], input_columns=['label'])

# 3. 拆分和打包 [cite: 288, 290-291]
imdb_train, imdb_valid = imdb_train.split([0.7, 0.3])
imdb_train = imdb_train.batch(64, drop_remainder=True)
imdb_valid = imdb_valid.batch(64, drop_remainder=True)

print("数据集已重置并处理完成。现在可以重新开始训练循环了。")

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:06:26.986.260 [mindspore/dataset/engine/datasets.py:1203] Dataset is shuffled before split.


数据集已重置并处理完成。现在可以重新开始训练循环了。


In [45]:
class CNN(nn.Cell):
    def __init__(self, embeddings, out_channels, kernel_size, output_dim, dropout, pad_idx):
        super().__init__()
        vocab_size, embedding_dim = embeddings.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, 
                                     embedding_table=Tensor(embeddings), 
                                     padding_idx=pad_idx)
        # 使用一维卷积提取文本特征 [cite: 25]
        self.conv = nn.Conv1d(in_channels=embedding_dim, 
                              out_channels=out_channels, 
                              kernel_size=kernel_size, 
                              has_bias=True)
        self.relu = nn.ReLU()
        self.max_pool = nn.AdaptiveMaxPool1d(1) # 池化层获取全局特征 
        self.fc = nn.Dense(out_channels, output_dim)
        self.dropout = nn.Dropout(1 - dropout)
        self.sigmoid = ops.Sigmoid()

    def construct(self, inputs):
        x = self.embedding(inputs) # [batch, seq_len, embed_dim]
        # 维度转换以适配 Conv1d: [batch, embed_dim, seq_len]
        x = x.transpose(0, 2, 1) 
        x = self.relu(self.conv(x))
        x = self.max_pool(x).squeeze(2) 
        x = self.dropout(x)
        return self.sigmoid(self.fc(x))

In [46]:
# 设置 CNN 的超参数
out_channels = 256      # 实验要求：改变卷积核数量进行对比
kernel_size = 3         # 卷积核大小
output_size = 1
dropout_rate = 0.5      # 实验要求：改变正则化手段 (Dropout) 进行对比
pad_idx = vocab.tokens_to_ids('<pad>')

# 实例化模型：注意这里用的是你新定义的 CNN 类
model = CNN(embeddings, out_channels, kernel_size, output_size, dropout_rate, pad_idx)

# 重新定义损失函数和优化器
loss_fn = nn.BCEWithLogitsLoss(reduction='mean')
optimizer = nn.Adam(model.trainable_params(), learning_rate=0.001)

print("CNN 模型实例化完成，优化器已重置。")

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:09:36.480.577 [mindspore/nn/layer/basic.py:173] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


CNN 模型实例化完成，优化器已重置。


In [63]:
# 1. 重新实例化 CNN 模型 [cite: 22, 30]
out_channels = 256
kernel_size = 3
output_size = 1
dropout_rate = 0.5 # 建议：将 keep_prob 改为 p=0.5 消除警告
pad_idx = vocab.tokens_to_ids('<pad>')

model = CNN(embeddings, out_channels, kernel_size, output_size, dropout_rate, pad_idx)

# 2. 重新定义损失函数和优化器 [cite: 350-351]
loss_fn = nn.BCEWithLogitsLoss(reduction='mean')
optimizer = nn.Adam(model.trainable_params(), learning_rate=0.001)

# 3. 核心：必须重新定义这些函数，让它们绑定到新的 CNN 模型上 [cite: 358-366]
def forward_fn(data, label):
    logits = model(data)
    loss = loss_fn(logits, label)
    return loss

# 重新生成梯度计算函数
grad_fn = ms.value_and_grad(forward_fn, None, optimizer.parameters)

def train_step(data, label):
    loss, grads = grad_fn(data, label)
    optimizer(grads)
    return loss

print("CNN 训练环境已彻底重置，现在可以开始训练了！")

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:04.696.743 [mindspore/nn/layer/basic.py:173] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


CNN 训练环境已彻底重置，现在可以开始训练了！


In [64]:
num_epochs = 5
best_valid_loss = float('inf')
ckpt_file_name = os.path.join(cache_dir, 'sentiment-analysis.ckpt')
for epoch in range(num_epochs):
    train_one_epoch(model, imdb_train, epoch)
    valid_loss = evaluate(model, imdb_valid, loss_fn, epoch)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        ms.save_checkpoint(model, ckpt_file_name)

Epoch 0:   0%|          | 0/273 [00:00<?, ?it/s][WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:10.388.121 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:10.389.730 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:10.395.803 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:10.413.054 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[ERROR] CORE(3526,ffff9f2320b0,python):2026-05-07-21:23:10.665.078 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_3526/1633872606.py

/

[WARNING] GE_ADPT(3526,ffff9f2320b0,python):2026-05-07-21:23:11.616.606 [mindspore/ccsrc/transform/graph_ir/convert.cc:885] ConvertAllNode] Parameter @kernel_graph_9:param_gradients cannot found in toposort lists.
[WARNING] DEVICE(3526,ffff9f2320b0,python):2026-05-07-21:23:11.625.683 [mindspore/ccsrc/plugin/device/ascend/hal/hardware/ge_graph_executor.cc:1258] RunInitGraph] Can not find init_subgraph.kernel_graph_9 sub graph, don't need data init subgraph in INFER mode.
Epoch 0:   0%|          | 0/117 [00:00<?, ?it/s][WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:28.323.392 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:23:28.338.053 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[ERROR] CORE(3526,ffff9f2320b0,python):2026-05-07-21:23:28.438.200 [mindspore/core/utils/fi

-

[WARNING] DEVICE(3526,ffff9f2320b0,python):2026-05-07-21:23:28.759.381 [mindspore/ccsrc/plugin/device/ascend/hal/hardware/ge_graph_executor.cc:1258] RunInitGraph] Can not find init_subgraph.kernel_graph_11 sub graph, don't need data init subgraph in INFER mode.
Epoch 4: 100%|██████████| 117/117 [00:10<00:00, 11.18it/s, acc=0.941, loss=0.534]


In [65]:
param_dict = ms.load_checkpoint(ckpt_file_name)
ms.load_param_into_net(model, param_dict)
imdb_test = imdb_test.batch(64)
evaluate(model, imdb_test, loss_fn)

Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s][ERROR] MD(3526,fff8037fe1e0,python):2026-05-07-21:25:08.137.132 [mindspore/ccsrc/minddata/dataset/util/task_manager.cc:227] InterruptMaster] MindSpore dataset is terminated with err msg: Exception thrown from dataset pipeline. Refer to 'Dataset Pipeline Error Message'. Inconsistent batch shapes, batch operation expects same shape for each data row, but got inconsistent shape in column 0, expected shape for this column is:<64,500>, got shape:<40,500>
Line of code : 216
File         : mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc

Epoch 0:   0%|          | 0/1 [00:09<?, ?it/s]


RuntimeError: Exception thrown from dataset pipeline. Refer to 'Dataset Pipeline Error Message'. 

------------------------------------------------------------------
- Dataset Pipeline Error Message: 
------------------------------------------------------------------
[ERROR] Inconsistent batch shapes, batch operation expects same shape for each data row, but got inconsistent shape in column 0, expected shape for this column is:<64,500>, got shape:<40,500>.

------------------------------------------------------------------
- C++ Call Stack: (For framework developers) 
------------------------------------------------------------------
mindspore/ccsrc/minddata/dataset/engine/datasetops/batch_op.cc(216).




In [57]:
score_map = {
    1: "Positive",
    0: "Negative"
}

def predict_sentiment(model, vocab, sentence):
    model.set_train(False)
    tokenized = sentence.lower().split()
    indexed = vocab.tokens_to_ids(tokenized)
    tensor = ms.Tensor(indexed, ms.int32)
    tensor = tensor.expand_dims(0)
    prediction = model(tensor)
    return score_map[int(np.round(ops.sigmoid(prediction).asnumpy()))]

In [52]:
predict_sentiment(model, vocab, "This film is terrible")

[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:15:50.845.716 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(3526:281473351622832,MainProcess):2026-05-07-21:15:50.857.087 [mindspore/nn/layer/basic.py:199] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[ERROR] CORE(3526,ffff9f2320b0,python):2026-05-07-21:15:50.958.204 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_3526/2319351174.py]
[ERROR] CORE(3526,ffff9f2320b0,python):2026-05-07-21:15:50.958.270 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_3526/2319351174.py]
[WARNING] DEVICE(3526,ffff9f2320b0,python):2026-05-07-21:15:50.979.765 [mindspore/ccsrc/plugin/device/ascend/hal/hardware/ge_graph_executor.cc:1258] RunInitGraph] Can not find init_subgraph.kernel_graph_6 sub graph, don't need data init subgraph in I

|

'Positive'

In [60]:
predict_sentiment(model, vocab, "This film is good")

'Positive'